<a href="https://colab.research.google.com/github/akwve/STA160_Group2_Project/blob/main/ASK_PEOPLE_ABOUT_TWEETS_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
base_dir = "/content/bert_fake_news_model"
model_dir = os.path.join(base_dir, "content", "bert_fake_news_model")
print("Model directory:", model_dir)
print("Files:", os.listdir(model_dir))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
model.eval()
while True:
    text = input("\nEnter text to classify (or type 'quit' to exit): ")
    if text.lower() == "quit":
        break
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred = probs.argmax(dim=-1).item()
    label = "Fake" if pred == 0 else "Real"
    confidence = probs[0][pred].item()
    print(f"→ Prediction: {label} (confidence: {confidence:.3f})")

Model directory: /content/bert_fake_news_model/content/bert_fake_news_model
Files: ['tokenizer.json', 'special_tokens_map.json', 'config.json', 'model.safetensors', 'vocab.txt', 'tokenizer_config.json']
Using device: cpu

Enter text to classify (or type 'quit' to exit): Rugby is a NAZI sport
→ Prediction: Fake (confidence: 0.998)

Enter text to classify (or type 'quit' to exit): quit


In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import shutil, os, zipfile

SAVE_DIR = "/content/bert_fake_news_model"

# If your objects are already in memory:
# model.save_pretrained(SAVE_DIR)
# tokenizer.save_pretrained(SAVE_DIR)

# If you only have a path to your trained folder, you can skip saving again.
# Just make sure that folder contains at least: config.json, pytorch_model.bin (or safetensors), tokenizer.json/tokenizer files, special_tokens_map.json, vocab files.

# Zip for download
ZIP_PATH = "/content/bert_fake_news_model.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(SAVE_DIR):
        for f in files:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, SAVE_DIR)
            z.write(full, os.path.join("bert_fake_news_model", rel))

print("Zipped to:", ZIP_PATH)


Zipped to: /content/bert_fake_news_model.zip
